# Avance 3: integración, análisis descriptivo y diseño de un ETL reproducible
**Proyecto:** Analítica de Incidentes Viales CDMX (C4/C5)  
**Equipo:** Las Galácticas  
**Propósito:** integrar tres archivos CSV de incidentes viales, transformarlos, validarlos y guardar los datos preparados con un ETL reproducible y una bitácora automática; calcular medidas descriptivas para la pregunta de negocio.  
**Pregunta de negocio:** ¿En qué alcaldías, días de la semana y franjas horarias se concentra la mayor cantidad de incidentes viales y cuáles son los tipos de incidentes más frecuentes para priorizar la asignación preventiva de recursos de emergencia?  
**Fuente:** Kaggle, "Mexico Road Accidents during 2017-2019" (CSV), descargada el 04/10/2026.  
**Archivos crudos (no se modifican):**  
- data/raw/incidentes-viales-c5-2017.csv  
- data/raw/incidentes-viales-c5-2018.csv  
- data/raw/incidentes-viales-c5-2019.csv  

**Salidas:** data/processed/ (datos preparados y pendientes) y bitacora/bitacora.csv  
**Orden de ejecución:** de arriba hacia abajo, sin correcciones manuales.  
**Secciones:** 0. Configuración · 1. Extraer · 2. Integrar · 3. Transformar · 4. Validar · 5. Guardar · 6. Conciliación · 7. Resumen · 8. Estadística descriptiva · 9. Reproducibilidad

## 0. Configuración
Objetivo: conectar Drive, definir rutas de entrada y salida, y preparar la función `anotar()` que genera la bitácora automática.

In [57]:
# Configuración: librerías, rutas de entrada (crudos) y salida, y función anotar() para la bitácora automática.
from google.colab import drive
from pathlib import Path
from datetime import datetime
import pandas as pd
from zoneinfo import ZoneInfo  # Zona horaria de México para la fecha de la bitácora

drive.mount('/content/drive')

# Rutas de entrada: archivos crudos (no se modifican)
base = Path("/content/drive/MyDrive/BI S&R/Datos/Kaggle")
rutas = {
    2017: base / "incidentes-viales-c5-2017.csv",
    2018: base / "incidentes-viales-c5-2018.csv",
    2019: base / "incidentes-viales-c5-2019.csv",
}

# Rutas de salida: se crean si no existen
proyecto = Path("/content/drive/MyDrive/BI S&R")
carpeta_processed = proyecto / "data" / "processed"
carpeta_bitacora = proyecto / "bitacora"
carpeta_processed.mkdir(parents=True, exist_ok=True)
carpeta_bitacora.mkdir(parents=True, exist_ok=True)

# Bitácora automática (adaptada de la Nota 11)
bitacora = []

def anotar(archivo, paso, filas_antes, filas_despues, detalle,
           regla, resultado, observaciones="", responsable="Sarai, Rosa Isela"):
    bitacora.append({
        "fecha": datetime.now(ZoneInfo("America/Mexico_City")).isoformat(),
        "archivo": str(archivo),
        "paso": paso,
        "filas_antes": filas_antes,
        "filas_despues": filas_despues,
        "detalle": detalle,
        "responsable": responsable,
        "regla": regla,
        "resultado": resultado,
        "observaciones": observaciones,
    })

for anio, ruta in rutas.items():
    print(anio, "Encontrado" if ruta.exists() else "No encontrado")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2017 Encontrado
2018 Encontrado
2019 Encontrado


## 1. Extraer
Objetivo: leer los tres CSV crudos sin modificarlos y comprobar que tengan las 18 columnas esperadas, en el mismo orden.

In [58]:
# Extracción: lee cada CSV, verifica sus columnas contra el contrato de entrada y anota el paso en la bitácora.
columnas_esperadas = [
    "folio", "fecha_creacion", "hora_creacion", "dia_semana", "codigo_cierre",
    "fecha_cierre", "año_cierre", "mes_cierre", "hora_cierre", "delegacion_inicio",
    "incidente_c4", "latitud", "longitud", "clas_con_f_alarma", "tipo_entrada",
    "delegacion_cierre", "geopoint", "mes"
]

def extraer(ruta):
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró: {ruta}")
    tabla = pd.read_csv(ruta, sep=",", encoding="utf-8")
    if tabla.columns.tolist() != columnas_esperadas:
        raise ValueError(f"Columnas distintas a las esperadas en {ruta.name}")
    anotar(ruta.name, "extraer", len(tabla), len(tabla),
           f"{tabla.shape[1]} columnas leídas",
           "Verificar 18 columnas y su orden",
           "Correcto", "Archivo crudo conservado sin cambios")
    return tabla

tablas = {anio: extraer(ruta) for anio, ruta in rutas.items()}

for anio, tabla in tablas.items():
    print(anio, tabla.shape)

2017 (226535, 18)
2018 (246577, 18)
2019 (207634, 18)


## 2. Integrar
Objetivo: unir los tres lotes anuales con `concat`, porque tienen la misma estructura. Cada fila representa un incidente vial (un folio), antes y después de la unión. Se agrega `archivo_origen` para saber de qué archivo viene cada fila, y se comprueba que no se pierdan ni se dupliquen registros.

In [59]:
# Integración: concat de los 3 lotes, con archivo_origen para la trazabilidad; se comprueban la cobertura y la unicidad del folio.
def integrar(tablas):
    integrada = pd.concat(
        [t.assign(archivo_origen=anio) for anio, t in tablas.items()],
        ignore_index=True
    )
    suma_lotes = sum(len(t) for t in tablas.values())
    folios_rep = integrada["folio"].duplicated().sum()

    if len(integrada) != suma_lotes:
        raise ValueError("La unión no conserva el total de filas de los lotes")

    anotar("3 CSV (2017-2019)", "integrar", suma_lotes, len(integrada),
           "Unión de lotes anuales con archivo_origen",
           "concat de tablas con la misma estructura; una fila = un folio",
           "Correcto" if folios_rep == 0 else "Con observaciones",
           f"Folios repetidos entre lotes: {folios_rep}")
    return integrada

incidentes = integrar(tablas)

print("filas integradas:", incidentes.shape)
print("filas por lote:")
print(incidentes["archivo_origen"].value_counts().sort_index())
print("folios repetidos:", incidentes["folio"].duplicated().sum())

filas integradas: (680746, 19)
filas por lote:
archivo_origen
2017    226535
2018    246577
2019    207634
Name: count, dtype: int64
folios repetidos: 0


## 3. Transformar
Objetivo: trabajar sobre una copia (`limpio`) y aplicar las reglas del Avance 2:
- **R1 (fechas):** `fecha_creacion` y `fecha_cierre` pasan de texto a fecha (vienen como dd/mm/aaaa y dd/mm/aa).
- **R2 (horas):** `hora_creacion` y `hora_cierre` pasan a tipo hora.
- **R3 (texto):** `incidente_c4` se pasa a minúsculas y sin espacios (de 23 a 22 tipos).

Además se crean variables nuevas para la pregunta de negocio: `hora` y `franja_horaria` (madrugada, mañana, tarde, noche), y las marcas de casos a revisar, sin corregir valores.

In [60]:
# Transformación: R1 fechas, R2 horas, R3 texto, variables derivadas y marcas; cada regla se anota en la bitácora.
def convertir_fecha(serie):
    f4 = pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce")
    f2 = pd.to_datetime(serie, format="%d/%m/%y", errors="coerce")
    return f4.fillna(f2)

def transformar(incidentes):
    limpio = incidentes.copy()
    n = len(limpio)

    # R1: fechas
    fallos = 0
    for col in ["fecha_creacion", "fecha_cierre"]:
        convertida = convertir_fecha(limpio[col])
        fallos += (limpio[col].notna() & convertida.isna()).sum()
        limpio[col] = convertida
    anotar("incidentes", "R1_fechas", n, len(limpio),
           "fecha_creacion y fecha_cierre a tipo fecha",
           "Formatos dd/mm/aaaa y dd/mm/aa",
           "Correcto" if fallos == 0 else "Con observaciones",
           f"Fallos de conversión: {fallos}")

    # R2: horas
    fallos = 0
    for col in ["hora_creacion", "hora_cierre"]:
        convertida = pd.to_datetime(limpio[col], format="%H:%M:%S", errors="coerce")
        fallos += (limpio[col].notna() & convertida.isna()).sum()
        limpio[col] = convertida.dt.time
        if col == "hora_creacion":
            limpio["hora"] = convertida.dt.hour   # variable derivada
    anotar("incidentes", "R2_horas", n, len(limpio),
           "hora_creacion y hora_cierre a tipo hora; se deriva hora (0-23)",
           "Formato HH:MM:SS",
           "Correcto" if fallos == 0 else "Con observaciones",
           f"Fallos de conversión: {fallos}")

    # R3: texto de incidente_c4
    antes = limpio["incidente_c4"].nunique()
    limpio["incidente_c4"] = limpio["incidente_c4"].astype("string").str.strip().str.lower()
    despues = limpio["incidente_c4"].nunique()
    anotar("incidentes", "R3_texto", n, len(limpio),
           "incidente_c4 en minúsculas y sin espacios exteriores",
           "Unificar escritura de categorías",
           "Correcto", f"Tipos distintos: {antes} -> {despues}")

    # Variable derivada: franja horaria
    limpio["franja_horaria"] = pd.cut(
        limpio["hora"], bins=[0, 6, 12, 18, 24], right=False,
        labels=["Madrugada (00-05)", "Mañana (06-11)", "Tarde (12-17)", "Noche (18-23)"]
    )
    anotar("incidentes", "derivar_franja", n, len(limpio),
           "Se crea franja_horaria a partir de hora",
           "Cortes de 6 horas: 0-5, 6-11, 12-17, 18-23",
           "Correcto", f"Sin franja: {limpio['franja_horaria'].isna().sum()}")

    # Marcas de casos a revisar (no modifican valores)
    lat_ok = limpio["latitud"].between(19.09402002, 19.6322009)
    lon_ok = limpio["longitud"].between(-99.37056146, -98.92945008)
    limpio["coord_fuera_rango"] = (limpio["latitud"].notna() & ~lat_ok) | (limpio["longitud"].notna() & ~lon_ok)
    limpio["es_falso_f"] = limpio["codigo_cierre"].str.startswith("(F)")
    limpio["fecha_otro_anio"] = limpio["fecha_creacion"].dt.year != limpio["archivo_origen"]
    marcas = limpio[["coord_fuera_rango", "es_falso_f", "fecha_otro_anio"]].sum()
    anotar("incidentes", "marcar_casos", n, len(limpio),
           "Marcas True/False sin corregir valores",
           "Rango de coordenadas del diccionario, código F y año distinto al archivo",
           "Con observaciones",
           f"coord_fuera_rango={marcas.iloc[0]}, es_falso_f={marcas.iloc[1]}, fecha_otro_anio={marcas.iloc[2]}")
    return limpio

limpio = transformar(incidentes)

print("filas:", len(limpio), "| columnas:", limpio.shape[1])
print("tipos de incidente:", limpio["incidente_c4"].nunique())
print(limpio["franja_horaria"].value_counts().sort_index())
print(limpio[["coord_fuera_rango", "es_falso_f", "fecha_otro_anio"]].sum())

filas: 680746 | columnas: 24
tipos de incidente: 22
franja_horaria
Madrugada (00-05)     77530
Mañana (06-11)       141342
Tarde (12-17)        216351
Noche (18-23)        245523
Name: count, dtype: int64
coord_fuera_rango      13
es_falso_f           5244
fecha_otro_anio        49
dtype: int64


## 4. Validar
Objetivo: separar los registros en **válidos** y **pendientes**, y comprobar con controles que los válidos cumplan el contrato de salida.
- **R4 (alcaldía):** las filas sin `delegacion_inicio` pasan a *pendientes*, porque sin alcaldía no responden la pregunta de negocio. No se borran: se guardan aparte con su motivo.
- **Controles:** campos obligatorios completos, folio único, tipos de fecha y hora, catálogos (días, alcaldías y códigos de cierre), rango de fechas y franja horaria asignada.

In [61]:
# Validación: R4 separa válidos y pendientes; después se aplican controles por tabla sobre los válidos.
import datetime as dt

alcaldias = {"IZTAPALAPA", "GUSTAVO A. MADERO", "CUAUHTEMOC", "MIGUEL HIDALGO", "COYOACAN",
             "BENITO JUAREZ", "ALVARO OBREGON", "VENUSTIANO CARRANZA", "TLALPAN", "AZCAPOTZALCO",
             "IZTACALCO", "XOCHIMILCO", "TLAHUAC", "CUAJIMALPA", "MAGDALENA CONTRERAS", "MILPA ALTA"}
dias = {"Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"}
codigos = {"A", "D", "N", "F", "I"}
obligatorios = ["folio", "fecha_creacion", "hora_creacion", "dia_semana",
                "delegacion_inicio", "incidente_c4", "franja_horaria"]

def separar(limpio):
    sin_alcaldia = limpio["delegacion_inicio"].isna()
    validos = limpio[~sin_alcaldia].copy()
    pendientes = limpio[sin_alcaldia].copy()
    pendientes["motivo_pendiente"] = "R4: sin delegacion_inicio"
    anotar("limpio", "R4_separar", len(limpio), len(validos),
           "Separación en válidos y pendientes",
           "Sin delegacion_inicio no se puede ubicar el incidente",
           "Con observaciones", f"Pendientes: {len(pendientes)} (se guardan aparte)")
    return validos, pendientes

def controlar(t):
    no_es_hora = lambda c: (~t[c].map(lambda x: isinstance(x, dt.time))).sum()
    pruebas = {
        "Obligatorios sin vacíos": t[obligatorios].isna().any(axis=1).sum(),
        "Folio único": t["folio"].duplicated().sum(),
        "Fechas en tipo fecha": sum(not pd.api.types.is_datetime64_any_dtype(t[c])
                                    for c in ["fecha_creacion", "fecha_cierre"]),
        "Horas en tipo hora": no_es_hora("hora_creacion") + no_es_hora("hora_cierre"),
        "Día de la semana en catálogo": (~t["dia_semana"].isin(dias)).sum(),
        "Alcaldía de inicio en catálogo": (~t["delegacion_inicio"].isin(alcaldias)).sum(),
        "Código de cierre en catálogo": (~t["codigo_cierre"].str[1].isin(codigos)).sum(),
        "Fecha de creación 2016-12 a 2019": (~t["fecha_creacion"].between("2016-12-01", "2019-12-31")).sum(),
        "Hora entre 0 y 23": (~t["hora"].between(0, 23)).sum(),
    }
    controles = pd.DataFrame({"casos que no cumplen": pruebas})
    controles["cumple"] = controles["casos que no cumplen"] == 0
    fallidos = (~controles["cumple"]).sum()
    anotar("validos", "controles", len(t), len(t),
           f"{len(pruebas)} controles del contrato de salida",
           "Completitud, unicidad, tipos, catálogos y rangos",
           "Correcto" if fallidos == 0 else "Con observaciones",
           f"Controles no cumplidos: {fallidos}")
    return controles

validos, pendientes = separar(limpio)
controles = controlar(validos)

print("válidos:", len(validos), "| pendientes:", len(pendientes))
print(pendientes["archivo_origen"].value_counts().sort_index())
print(controles)

válidos: 680687 | pendientes: 59
archivo_origen
2017     4
2018    36
2019    19
Name: count, dtype: int64
                                  casos que no cumplen  cumple
Obligatorios sin vacíos                              0    True
Folio único                                          0    True
Fechas en tipo fecha                                 0    True
Horas en tipo hora                                   0    True
Día de la semana en catálogo                         0    True
Alcaldía de inicio en catálogo                       0    True
Código de cierre en catálogo                         0    True
Fecha de creación 2016-12 a 2019                     0    True
Hora entre 0 y 23                                    0    True


## 5. Guardar
Objetivo: exportar las salidas del proceso en CSV:
- `data/processed/incidentes_preparados.csv`: registros válidos.
- `data/processed/incidentes_pendientes.csv`: registros excluidos, con su motivo.
- `data/processed/controles.csv`: resultado de los controles.
- `bitacora/bitacora.csv`: bitácora automática del proceso.

Al final se vuelven a leer los archivos para comprobar que se guardaron completos.

In [62]:
# Guardado: exporta válidos, pendientes, controles y bitácora; comprueba los conteos releyendo los archivos.
salidas = {
    "preparados": carpeta_processed / "incidentes_preparados.csv",
    "pendientes": carpeta_processed / "incidentes_pendientes.csv",
    "controles": carpeta_processed / "controles.csv",
}

def guardar_bitacora():
    ruta = carpeta_bitacora / "bitacora.csv"
    pd.DataFrame(bitacora).to_csv(ruta, index=False, encoding="utf-8")
    return ruta

def guardar(validos, pendientes, controles):
    validos.to_csv(salidas["preparados"], index=False, encoding="utf-8")
    pendientes.to_csv(salidas["pendientes"], index=False, encoding="utf-8")
    controles.to_csv(salidas["controles"], encoding="utf-8")

    filas_prep = len(pd.read_csv(salidas["preparados"], usecols=["folio"]))
    filas_pend = len(pd.read_csv(salidas["pendientes"], usecols=["folio"]))
    ok = filas_prep == len(validos) and filas_pend == len(pendientes)

    anotar("data/processed", "guardar", len(validos) + len(pendientes), filas_prep + filas_pend,
           "Exportación de preparados, pendientes y controles",
           "CSV utf-8 sin índice; se releen para comprobar conteos",
           "Correcto" if ok else "Con observaciones",
           f"Preparados: {filas_prep} | Pendientes: {filas_pend}")
    return filas_prep, filas_pend

filas_prep, filas_pend = guardar(validos, pendientes, controles)
ruta_bitacora = guardar_bitacora()

print("preparados guardados:", filas_prep)
print("pendientes guardados:", filas_pend)
print("bitácora:", ruta_bitacora)
print(pd.read_csv(ruta_bitacora)[["paso", "filas_antes", "filas_despues", "resultado"]])

preparados guardados: 680687
pendientes guardados: 59
bitácora: /content/drive/MyDrive/BI S&R/bitacora/bitacora.csv
              paso  filas_antes  filas_despues          resultado
0          extraer       226535         226535           Correcto
1          extraer       246577         246577           Correcto
2          extraer       207634         207634           Correcto
3         integrar       680746         680746           Correcto
4        R1_fechas       680746         680746           Correcto
5         R2_horas       680746         680746           Correcto
6         R3_texto       680746         680746           Correcto
7   derivar_franja       680746         680746           Correcto
8     marcar_casos       680746         680746  Con observaciones
9       R4_separar       680746         680687  Con observaciones
10       controles       680687         680687           Correcto
11         guardar       680746         680746           Correcto


## 6. Conciliación
Objetivo: explicar el destino de cada registro original y comprobar que los conteos cuadran:
**originales = repetidos retirados + válidos + pendientes**.
Se usan solo las tablas finales (válidos y pendientes); `incidentes` y `limpio` no se suman otra vez porque representan los mismos casos en etapas anteriores.

In [63]:
# Conciliación: destino de los registros originales, total y por archivo, sin sumar dos veces la misma etapa.
def conciliar(tablas, incidentes, validos, pendientes):
    originales = sum(len(t) for t in tablas.values())
    repetidos = originales - len(incidentes)   # en este proyecto no se retiraron filas repetidas

    total = pd.Series({
        "originales (3 CSV)": originales,
        "repetidos retirados": repetidos,
        "válidos": len(validos),
        "pendientes": len(pendientes),
    })
    cuadra = originales == repetidos + len(validos) + len(pendientes)

    # Ningún folio puede estar en válidos y en pendientes a la vez
    cruce = len(set(validos["folio"]) & set(pendientes["folio"]))

    por_archivo = pd.DataFrame({
        "originales": {anio: len(t) for anio, t in tablas.items()},
        "válidos": validos["archivo_origen"].value_counts(),
        "pendientes": pendientes["archivo_origen"].value_counts(),
    }).fillna(0).astype(int)
    por_archivo["cuadra"] = por_archivo["originales"] == por_archivo["válidos"] + por_archivo["pendientes"]

    anotar("validos + pendientes", "conciliar", originales, len(validos) + len(pendientes),
           f"{originales} = {repetidos} + {len(validos)} + {len(pendientes)}",
           "originales = repetidos + válidos + pendientes",
           "Correcto" if cuadra and cruce == 0 else "Con observaciones",
           f"Folios en ambas tablas: {cruce}")
    return total, cuadra, cruce, por_archivo

total, cuadra, cruce, por_archivo = conciliar(tablas, incidentes, validos, pendientes)

print(total)
print("¿cuadra?", cuadra, "| folios en ambas tablas:", cruce)
print(por_archivo)

originales (3 CSV)     680746
repetidos retirados         0
válidos                680687
pendientes                 59
dtype: int64
¿cuadra? True | folios en ambas tablas: 0
      originales  válidos  pendientes  cuadra
2017      226535   226531           4    True
2018      246577   246541          36    True
2019      207634   207615          19    True


## 7. Resumen por alcaldía, día y franja horaria
Objetivo: agrupar los incidentes válidos por alcaldía, día de la semana y franja horaria para responder la pregunta de negocio.
**Cambio de granularidad:** antes, una fila = un incidente; después, una fila = una combinación de alcaldía, día y franja. Se comprueba que la suma de incidentes del resumen sea igual al total de válidos, para no perder ni duplicar casos.

In [64]:
# Resumen: groupby por alcaldía, día y franja (estilo Nota 09); se valida que la suma coincida con los válidos.
def resumir(validos):
    resumen = (
        validos
        .groupby(["delegacion_inicio", "dia_semana", "franja_horaria"],
                 dropna=False, as_index=False, observed=True)
        .agg(
            incidentes=("folio", "size"),
            falsas_alarmas=("es_falso_f", "sum")
        )
        .sort_values("incidentes", ascending=False)
        .reset_index(drop=True)
    )
    cuadra = resumen["incidentes"].sum() == len(validos)

    anotar("validos", "resumir", len(validos), len(resumen),
           "Agrupación por alcaldía, día y franja horaria",
           "Una fila = una combinación alcaldía-día-franja; suma = total de válidos",
           "Correcto" if cuadra else "Con observaciones",
           f"Combinaciones: {len(resumen)} | Suma de incidentes: {resumen['incidentes'].sum()}")
    resumen.to_csv(carpeta_processed / "resumen_alcaldia_dia_franja.csv", index=False, encoding="utf-8")
    return resumen, cuadra

resumen, cuadra = resumir(validos)

print("combinaciones:", len(resumen), "| ¿suma = válidos?", cuadra)
print(resumen.head(10))

combinaciones: 448 | ¿suma = válidos? True
   delegacion_inicio dia_semana franja_horaria  incidentes  falsas_alarmas
0         IZTAPALAPA    Viernes  Noche (18-23)        7528              84
1         IZTAPALAPA     Sábado  Noche (18-23)        6910              68
2         IZTAPALAPA    Domingo  Noche (18-23)        5833              75
3         IZTAPALAPA     Jueves  Noche (18-23)        5314              50
4         IZTAPALAPA  Miércoles  Noche (18-23)        5287              53
5         IZTAPALAPA     Sábado  Tarde (12-17)        5287              71
6         IZTAPALAPA     Martes  Noche (18-23)        5207              48
7  GUSTAVO A. MADERO    Viernes  Noche (18-23)        5153              98
8         IZTAPALAPA    Viernes  Tarde (12-17)        5135              54
9         IZTAPALAPA      Lunes  Noche (18-23)        5047              55


## 8. Estadística descriptiva
Objetivo: calcular cinco medidas sobre los datos **preparados** (válidos) para responder la pregunta de negocio.
- **M1:** frecuencia y porcentaje de incidentes por alcaldía.
- **M2:** media, mediana y desviación estándar de incidentes diarios, por día de la semana.
- **M3:** frecuencia y porcentaje por franja horaria.
- **M4:** los cinco tipos de incidente más frecuentes.
- **M5:** media, mediana, desviación estándar y percentil 90 de incidentes diarios por alcaldía.

Los promedios diarios (M2 y M5) usan 01/01/2017 a 30/11/2019, el periodo con cobertura completa; los días sin incidentes cuentan como 0.

In [65]:
# Estadística descriptiva sobre los datos preparados (válidos): M1 a M5.
def estadistica(validos):
    total = len(validos)

    # M1: frecuencia y porcentaje por alcaldía
    m1 = validos["delegacion_inicio"].value_counts().to_frame("incidentes")
    m1["porcentaje"] = (m1["incidentes"] / total * 100).round(1)

    # Serie diaria: periodo con cobertura completa; los días sin incidentes valen 0
    fechas = pd.date_range("2017-01-01", "2019-11-30", freq="D")
    en_periodo = validos[validos["fecha_creacion"].between(fechas[0], fechas[-1])]
    diario = en_periodo.groupby("fecha_creacion").size().reindex(fechas, fill_value=0)

    # M2: incidentes diarios por día de la semana
    nombres = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves",
               4: "Viernes", 5: "Sábado", 6: "Domingo"}
    m2 = diario.groupby(diario.index.dayofweek).agg(["mean", "median", "std"]).round(1)
    m2.index = m2.index.map(nombres)
    m2.columns = ["media", "mediana", "desv_est"]

    # M3: frecuencia y porcentaje por franja horaria
    m3 = validos["franja_horaria"].value_counts().sort_index().to_frame("incidentes")
    m3["porcentaje"] = (m3["incidentes"] / total * 100).round(1)

    # M4: cinco tipos de incidente más frecuentes
    m4 = validos["incidente_c4"].value_counts().head(5).to_frame("incidentes")
    m4["porcentaje"] = (m4["incidentes"] / total * 100).round(1)

    # M5: incidentes diarios por alcaldía (media, mediana, desviación estándar y percentil 90)
    por_alcaldia = (en_periodo.groupby(["delegacion_inicio", "fecha_creacion"]).size()
                    .unstack(fill_value=0).reindex(columns=fechas, fill_value=0))
    m5 = pd.DataFrame({
        "media": por_alcaldia.mean(axis=1),
        "mediana": por_alcaldia.median(axis=1),
        "desv_est": por_alcaldia.std(axis=1),
        "p90": por_alcaldia.quantile(0.9, axis=1),
    }).round(1).sort_values("media", ascending=False)

    anotar("validos", "estadistica", total, total,
           "Cinco medidas descriptivas (M1 a M5) sobre los datos preparados",
           "Frecuencias, porcentajes, media, mediana, desviación estándar y percentil 90",
           "Correcto",
           f"Días analizados: {len(fechas)} | Incidentes en el periodo: {int(diario.sum())}",)
    return m1, m2, m3, m4, m5

m1, m2, m3, m4, m5 = estadistica(validos)

print("M1 por alcaldía:\n", m1, "\n")
print("M2 incidentes diarios por día de la semana:\n", m2, "\n")
print("M3 por franja horaria:\n", m3, "\n")
print("M4 tipos de incidente más frecuentes:\n", m4, "\n")
print("M5 incidentes diarios por alcaldía:\n", m5)

M1 por alcaldía:
                      incidentes  porcentaje
delegacion_inicio                          
IZTAPALAPA               111344        16.4
GUSTAVO A. MADERO         69597        10.2
CUAUHTEMOC                67755        10.0
MIGUEL HIDALGO            56819         8.3
COYOACAN                  54124         8.0
BENITO JUAREZ             52070         7.6
ALVARO OBREGON            51760         7.6
VENUSTIANO CARRANZA       45682         6.7
TLALPAN                   43862         6.4
AZCAPOTZALCO              29146         4.3
IZTACALCO                 27741         4.1
XOCHIMILCO                26409         3.9
TLAHUAC                   16889         2.5
CUAJIMALPA                13774         2.0
MAGDALENA CONTRERAS       10143         1.5
MILPA ALTA                 3572         0.5 

M2 incidentes diarios por día de la semana:
            media  mediana  desv_est
Lunes      587.2    585.0     110.2
Martes     600.8    599.5     102.0
Miércoles  616.1    624.0     102.6

## 9. Reproducibilidad
Objetivo: repetir el proceso completo desde los archivos crudos, sin correcciones manuales, y comprobar que con la misma entrada y las mismas reglas se obtiene la misma tabla preparada. Cambian la fecha de la bitácora, pero no los datos.

In [66]:
# Reproducibilidad: se repite el ETL desde los CSV crudos y se compara con la primera ejecución.
import hashlib

def huella(tabla):
    # Código que resume el contenido de la tabla; si cambia algún dato, cambia el código
    texto = tabla.reset_index(drop=True).to_csv(index=False)
    return hashlib.md5(texto.encode("utf-8")).hexdigest()

pasos_antes = len(bitacora)

# Segunda ejecución completa, desde los archivos crudos
otras_tablas = {anio: extraer(ruta) for anio, ruta in rutas.items()}
otra_integrada = integrar(otras_tablas)
otro_limpio = transformar(otra_integrada)
otros_validos, otros_pendientes = separar(otro_limpio)

# Se quitan las anotaciones de la repetición para no duplicar pasos en la bitácora
del bitacora[pasos_antes:]

huella_validos_1 = huella(validos)
huella_validos_2 = huella(otros_validos)
huella_pend_1 = huella(pendientes)
huella_pend_2 = huella(otros_pendientes)

reproducible = (huella_validos_1 == huella_validos_2) and (huella_pend_1 == huella_pend_2)

anotar("data/processed", "reproducibilidad", len(validos), len(otros_validos),
       "Repetición del ETL desde los CSV crudos y comparación con la primera ejecución",
       "Misma entrada y mismas reglas producen la misma tabla preparada",
       "Correcto" if reproducible else "Con observaciones",
       f"Huella válidos: {huella_validos_1}",)

print("Filas válidas 1ª / 2ª ejecución:", len(validos), "/", len(otros_validos))
print("Filas pendientes 1ª / 2ª ejecución:", len(pendientes), "/", len(otros_pendientes))
print("Huella válidos 1ª:", huella_validos_1)
print("Huella válidos 2ª:", huella_validos_2)
print("¿Reproducible?", reproducible)

Filas válidas 1ª / 2ª ejecución: 680687 / 680687
Filas pendientes 1ª / 2ª ejecución: 59 / 59
Huella válidos 1ª: 99fabc4ea38f98671fa9a4e7e637ea21
Huella válidos 2ª: 99fabc4ea38f98671fa9a4e7e637ea21
¿Reproducible? True


In [67]:
# Guarda la bitácora completa al final del proceso y muestra los pasos registrados.
ruta_bitacora = guardar_bitacora()
print("BITÁCORA FINAL | pasos:", len(bitacora))
print(pd.read_csv(ruta_bitacora)[["paso", "responsable", "resultado"]])

BITÁCORA FINAL | pasos: 16
                paso        responsable          resultado
0            extraer  Sarai, Rosa Isela           Correcto
1            extraer  Sarai, Rosa Isela           Correcto
2            extraer  Sarai, Rosa Isela           Correcto
3           integrar  Sarai, Rosa Isela           Correcto
4          R1_fechas  Sarai, Rosa Isela           Correcto
5           R2_horas  Sarai, Rosa Isela           Correcto
6           R3_texto  Sarai, Rosa Isela           Correcto
7     derivar_franja  Sarai, Rosa Isela           Correcto
8       marcar_casos  Sarai, Rosa Isela  Con observaciones
9         R4_separar  Sarai, Rosa Isela  Con observaciones
10         controles  Sarai, Rosa Isela           Correcto
11           guardar  Sarai, Rosa Isela           Correcto
12         conciliar  Sarai, Rosa Isela           Correcto
13           resumir  Sarai, Rosa Isela           Correcto
14       estadistica  Sarai, Rosa Isela           Correcto
15  reproducibilidad  Sarai, 